# 4. Feature Engineering

### Weather data

Calculate monthly averages per region (mean of all Jans, Febs, etc.), number of heatwave days, drought index, 

Load weather data parquet

In [1]:
import pandas as pd

In [2]:
weather = pd.read_parquet('../Data/Processed/weather_states.parquet')
weather.sample(10)

,date,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max,state
3012,2011-09-29 00:00:00+00:00,16.393002,22.668001,10.818000,0.000000,9.449572,Bremen
35392,2014-05-16 00:00:00+00:00,10.836085,15.359000,4.009000,0.000000,16.676977,Mecklenburg-Vorpommern
40426,2006-08-25 00:00:00+00:00,13.626167,16.884499,11.534500,7.100000,20.898611,Baden-Württemberg
58693,2013-08-25 00:00:00+00:00,17.112082,21.072500,14.122499,4.100000,20.730501,Thüringen
110253,2004-04-09 00:00:00+00:00,3.575667,9.259001,-1.591000,0.000000,13.854155,Bayern
91148,2016-06-25 00:00:00+00:00,18.282833,19.687000,14.887000,15.500000,17.577440,Saarland
122112,2015-03-27 00:00:00+00:00,5.251583,8.462000,3.362000,1.300000,24.149532,Hessen
57798,2011-03-14 00:00:00+00:00,8.641251,12.772500,5.722500,2.300000,8.534353,Thüringen
41397,2009-04-22 00:00:00+00:00,11.994918,16.534500,7.384500,0.000000,16.583460,Baden-Württemberg
97173,2011-06-22 00:00:00+00:00,17.656752,22.463001,13.513000,17.999998,26.749235,Niedersachsen


In [3]:
weather.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125680 entries, 0 to 125679
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype              
---  ------               --------------   -----              
 0   date                 125680 non-null  datetime64[ms, UTC]
 1   temperature_2m_mean  125680 non-null  float32            
 2   temperature_2m_max   125680 non-null  float32            
 3   temperature_2m_min   125680 non-null  float32            
 4   precipitation_sum    125680 non-null  float32            
 5   wind_speed_10m_max   125680 non-null  float32            
 6   state                125680 non-null  object             
dtypes: datetime64[ms, UTC](1), float32(5), object(1)
memory usage: 4.3+ MB


In [4]:
# Extract month and year from datetime

weather['month'] = weather['date'].dt.month
weather['year'] = weather['date'].dt.year
weather['monthyear'] = weather['month'].astype(str) + '_' + weather['year'].astype(str)
weather.sample(5)

,date,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max,state,month,year,monthyear
45680,2021-01-12 00:00:00+00:00,0.843750,2.250000,-4.250000,7.499999,31.896080,Baden-Württemberg,1,2021,1_2021
15082,2023-04-14 00:00:00+00:00,6.855083,12.915500,0.965500,1.300000,15.328561,Nordrhein-Westfalen,4,2023,4_2023
83282,2016-06-14 00:00:00+00:00,15.908752,19.902500,13.002501,4.200000,9.585739,Hamburg,6,2016,6_2016
103691,2007-10-24 00:00:00+00:00,8.209167,9.867499,6.667500,0.000000,24.280659,Schleswig-Holstein,10,2007,10_2007
45492,2020-07-08 00:00:00+00:00,18.854166,25.500000,11.400000,0.000000,17.643673,Baden-Württemberg,7,2020,7_2020


In [5]:
monthly = weather.groupby(['state', 'year', 'month']).agg(
    temp_mean = ('temperature_2m_mean', 'mean'),
    precip_sum = ('precipitation_sum', 'sum')
).reset_index()

monthly

,state,year,month,temp_mean,precip_sum
0,Baden-Württemberg,2003,7,19.441086,56.400002
1,Baden-Württemberg,2003,8,23.156878,18.400000
2,Baden-Württemberg,2003,9,14.583459,25.699999
3,Baden-Württemberg,2003,10,6.437322,91.799995
4,Baden-Württemberg,2003,11,5.571653,58.000000
...,...,...,...,...,...
4123,Thüringen,2024,8,19.984138,72.899994
4124,Thüringen,2024,9,15.468820,88.400002
4125,Thüringen,2024,10,11.061694,62.299999
4126,Thüringen,2024,11,4.483958,53.499996


In [6]:
# Compute baseline temperature and precipitation (long-term monthly mean per Bundesland)
baseline = monthly.groupby(['state', 'month']).agg(
    baseline_temp = ('temp_mean', 'mean'),
    baseline_precip = ('precip_sum', 'mean')
).reset_index()

baseline


,state,month,baseline_temp,baseline_precip
0,Baden-Württemberg,1,0.888582,73.585716
1,Baden-Württemberg,2,1.658827,58.023811
2,Baden-Württemberg,3,4.780743,60.757145
3,Baden-Württemberg,4,9.264582,61.628571
4,Baden-Württemberg,5,13.113695,91.942856
...,...,...,...,...
187,Thüringen,8,18.361685,84.522728
188,Thüringen,9,14.538055,62.849998
189,Thüringen,10,10.015421,59.663635
190,Thüringen,11,5.154050,56.386364


In [7]:
# Calculate flood risk as n days per month with heavy rain based on regional thresholds
# NB - threshold for heavy rain can differ between regions 
# (some regions may have ecological features better able to deal with more rain)

thresholds = weather.groupby(['state','month'])['precipitation_sum'].quantile(0.95).reset_index()
thresholds = thresholds.rename(columns={'precipitation_sum':'rain_thresh'})

weather = weather.merge(thresholds, on=['state','month'])

weather['heavy_rain'] = (weather['precipitation_sum'] > 
                                     weather['rain_thresh']).astype(int)

weather.sample(10)

,date,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max,state,month,year,monthyear,rain_thresh,heavy_rain
81730,2012-03-15 00:00:00+00:00,6.360834,11.002501,2.8525,0.0,9.511088,Hamburg,3,2012,3_2012,7.300000,0
33440,2009-01-10 00:00:00+00:00,0.409000,1.059000,-1.1410,0.0,13.722565,Mecklenburg-Vorpommern,1,2009,1_2009,8.300000,0
111700,2008-03-26 00:00:00+00:00,0.309000,5.059000,-3.1910,1.4,22.086521,Bayern,3,2008,3_2008,7.799999,0
108237,2020-04-04 00:00:00+00:00,5.837500,11.100000,1.3500,0.0,18.892282,Schleswig-Holstein,4,2020,4_2020,7.610000,0
111545,2007-10-23 00:00:00+00:00,2.731917,4.059000,1.7090,4.1,13.324863,Bayern,10,2007,10_2007,8.779999,0
93386,2022-08-11 00:00:00+00:00,23.989584,29.700001,17.6500,0.0,17.727943,Saarland,8,2022,8_2022,12.700000,0
67701,2016-10-21 00:00:00+00:00,6.793417,9.395500,4.2955,3.6,11.592894,Berlin,10,2016,10_2016,8.590000,0
54017,2022-05-09 00:00:00+00:00,14.318751,21.750000,5.7000,0.0,11.384198,Sachsen-Anhalt,5,2022,5_2022,9.450000,0
53406,2020-09-05 00:00:00+00:00,16.100000,18.549999,13.6500,3.0,14.021525,Sachsen-Anhalt,9,2020,9_2020,8.015000,0
60255,2017-12-04 00:00:00+00:00,1.702083,2.350000,0.0500,5.5,23.219301,Thüringen,12,2017,12_2017,8.100000,0


In [8]:
weather[weather['heavy_rain']==1].sample(10)

,date,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max,state,month,year,monthyear,rain_thresh,heavy_rain
101840,2024-04-01 00:00:00+00:00,9.787500,11.600000,8.050000,12.000001,22.007162,Niedersachsen,4,2024,4_2024,6.700000,1
32866,2007-06-16 00:00:00+00:00,16.163166,19.009001,14.709001,21.600002,24.842607,Mecklenburg-Vorpommern,6,2007,6_2007,10.200000,1
12637,2016-08-03 00:00:00+00:00,16.968750,18.150000,15.950001,15.799999,22.264771,Nordrhein-Westfalen,8,2016,8_2016,13.090000,1
124630,2022-02-16 00:00:00+00:00,6.093167,10.689000,3.839000,12.800002,31.688177,Hessen,2,2022,2_2022,10.400000,1
67458,2016-02-21 00:00:00+00:00,7.697584,10.545500,5.545500,9.699999,29.795301,Berlin,2,2016,2_2016,6.500000,1
82514,2014-05-08 00:00:00+00:00,10.987916,14.802500,9.002501,10.800001,20.466246,Hamburg,5,2014,5_2014,10.650000,1
66882,2014-07-25 00:00:00+00:00,19.720503,23.895500,17.295500,12.000001,16.676977,Berlin,7,2014,7_2014,11.890000,1
77922,2023-04-14 00:00:00+00:00,7.262501,9.150000,6.050000,8.500001,20.523155,Brandenburg,4,2023,4_2023,5.400000,1
59036,2014-08-03 00:00:00+00:00,20.295416,23.972500,16.972500,19.900002,8.404285,Thüringen,8,2014,8_2014,12.975001,1
92464,2020-02-01 00:00:00+00:00,10.518750,11.150000,9.500000,19.300001,30.113304,Saarland,2,2020,2_2020,12.440002,1


In [9]:
# aggregate mean monthly temp, max monthly temp, precipitation sum per month per Bundesland

weather_monthly = weather.groupby(['state', 'monthyear']).agg(
    year = ('year', 'min'),
    month = ('month', 'min'),
    mean_temp = ('temperature_2m_mean', 'mean'),
    max_temp = ('temperature_2m_max', 'max'),
    total_precip = ('precipitation_sum', 'sum'),
    n_hot_days=('temperature_2m_max', lambda x: (x > 30).sum()),
    heavy_rain_days=('heavy_rain','sum')
).reset_index()

weather_monthly.sample(10)

,state,monthyear,year,month,mean_temp,max_temp,total_precip,n_hot_days,heavy_rain_days
3509,Sachsen-Anhalt,5_2009,2009,5,14.175624,27.793499,61.599998,0,0
4077,Thüringen,7_2018,2018,7,20.540928,33.650002,30.900000,5,0
2109,Niedersachsen,12_2004,2004,12,2.900500,8.662999,40.400002,0,1
534,Berlin,10_2021,2021,10,10.503360,21.799999,43.099998,0,1
2588,Rheinland-Pfalz,10_2011,2011,10,9.944970,23.894499,18.300001,0,0
1651,Hessen,2_2020,2020,2,4.709259,14.689000,182.899994,0,5
1844,Mecklenburg-Vorpommern,11_2019,2019,11,5.810347,12.750000,61.099998,0,2
3192,Sachsen,2_2013,2013,2,-1.578571,6.125000,68.199997,0,1
3139,Sachsen,11_2024,2024,11,4.400070,14.050000,52.100002,0,1
3152,Sachsen,12_2015,2015,12,6.734140,13.175000,35.400002,0,0


In [10]:
# Merge baseline back to weather df
weather_monthly = weather_monthly.merge(baseline, on=['state', 'month'])

# Compute anomaly 
# for precip - calculate relative anomaly, e.g. -0.4 = 40% drier than normal
# for temp - stick with original scale (20C is not really "100% hotter" than 10C)
weather_monthly['temp_anomaly'] = (weather_monthly['mean_temp'] - weather_monthly['baseline_temp'])
weather_monthly['precip_prop_anomaly'] = (weather_monthly['total_precip'] - weather_monthly['baseline_precip'])/weather_monthly['baseline_precip']


In [11]:
# Compute drought index as temp anomaly - precip anomaly
# (use z-scores to get temp and precip on same scale)
weather_monthly['temp_anom_z'] = weather_monthly.groupby(['state', 'month'])['temp_anomaly'].transform(
    lambda x: (x - x.mean()) / x.std()
)

weather_monthly['precip_anom_z'] = weather_monthly.groupby(['state', 'month'])['precip_prop_anomaly'].transform(
    lambda x: (x - x.mean()) / x.std()
)

weather_monthly['drought_index'] = weather_monthly['temp_anom_z'] - weather_monthly['precip_anom_z']

weather_monthly.sample(5)

,state,monthyear,year,month,mean_temp,max_temp,total_precip,n_hot_days,heavy_rain_days,baseline_temp,baseline_precip,temp_anomaly,precip_prop_anomaly,temp_anom_z,precip_anom_z,drought_index
2974,Saarland,4_2011,2011,4,13.011374,24.886999,23.800001,0,0,9.976909,57.452381,3.034466,-0.585744,1.668148,-1.170618,2.838766
2584,Rheinland-Pfalz,10_2007,2007,10,9.088923,19.344500,18.299999,0,0,10.230147,57.563637,-1.141225,-0.682091,-0.748899,-1.475333,0.726433
4122,Thüringen,9_2019,2019,9,14.317986,24.400000,53.299999,0,1,14.538055,62.849998,-0.220069,-0.151949,-0.140291,-0.351839,0.211548
2694,Rheinland-Pfalz,3_2010,2010,3,4.154245,17.144499,38.599998,0,0,4.865086,53.947620,-0.710841,-0.284491,-0.409985,-0.565265,0.155280
2970,Saarland,4_2007,2007,4,13.373667,25.286999,3.000000,0,0,9.976909,57.452381,3.396758,-0.947783,1.867313,-1.894158,3.761471


Add lagged weather variables (1+2+3 month lag, 3+6 month rolling-mean lag)

In [12]:
# First sort weather_monthly by state, year, month

weather_monthly_sorted = weather_monthly.sort_values(['state', 'year', 'month']).copy()
weather_monthly_sorted.head()

,state,monthyear,year,month,mean_temp,max_temp,total_precip,n_hot_days,heavy_rain_days,baseline_temp,baseline_precip,temp_anomaly,precip_prop_anomaly,temp_anom_z,precip_anom_z,drought_index
192,Baden-Württemberg,7_2003,2003,7,19.441086,31.734499,56.400002,2,1,19.108015,80.540909,0.333071,-0.299735,0.218184,-0.621976,0.840159
214,Baden-Württemberg,8_2003,2003,8,23.156878,34.684498,18.400000,12,0,18.709295,79.786362,4.447582,-0.769384,2.221096,-2.022276,4.243373
236,Baden-Württemberg,9_2003,2003,9,14.583459,27.684500,25.699999,0,0,14.722257,57.586365,-0.138798,-0.553714,-0.091014,-1.327952,1.236938
0,Baden-Württemberg,10_2003,2003,10,6.437322,20.734499,91.799995,0,2,10.330837,61.754543,-3.893515,0.486530,-2.139184,1.295548,-3.434732
22,Baden-Württemberg,11_2003,2003,11,5.571653,15.334500,58.000000,0,1,5.168454,64.163643,0.403199,-0.096061,0.284581,-0.176771,0.461352


In [13]:
# 1 month lag
weather_monthly_sorted['temp_anom_z_lag1'] = weather_monthly_sorted.groupby('state')['temp_anom_z'].shift(1)
weather_monthly_sorted['precip_anom_z_lag1'] = weather_monthly_sorted.groupby('state')['precip_anom_z'].shift(1)
weather_monthly_sorted['n_hot_days_lag1'] = weather_monthly_sorted.groupby('state')['n_hot_days'].shift(1)
weather_monthly_sorted['drought_index_lag1'] = weather_monthly_sorted.groupby('state')['drought_index'].shift(1)
weather_monthly_sorted['heavy_rain_days_lag1'] = weather_monthly_sorted.groupby('state')['heavy_rain_days'].shift(1)
# 2 month lag
weather_monthly_sorted['temp_anom_z_lag2'] = weather_monthly_sorted.groupby('state')['temp_anom_z'].shift(2)
weather_monthly_sorted['precip_anom_z_lag2'] = weather_monthly_sorted.groupby('state')['precip_anom_z'].shift(2)
weather_monthly_sorted['n_hot_days_lag2'] = weather_monthly_sorted.groupby('state')['n_hot_days'].shift(2)
weather_monthly_sorted['drought_index_lag2'] = weather_monthly_sorted.groupby('state')['drought_index'].shift(2)
weather_monthly_sorted['heavy_rain_days_lag2'] = weather_monthly_sorted.groupby('state')['heavy_rain_days'].shift(2)
# 3 month lag
weather_monthly_sorted['temp_anom_z_lag3'] = weather_monthly_sorted.groupby('state')['temp_anom_z'].shift(3)
weather_monthly_sorted['precip_anom_z_lag3'] = weather_monthly_sorted.groupby('state')['precip_anom_z'].shift(3)
weather_monthly_sorted['n_hot_days_lag3'] = weather_monthly_sorted.groupby('state')['n_hot_days'].shift(3)
weather_monthly_sorted['drought_index_lag3'] = weather_monthly_sorted.groupby('state')['drought_index'].shift(3)
weather_monthly_sorted['heavy_rain_days_lag3'] = weather_monthly_sorted.groupby('state')['heavy_rain_days'].shift(3)



# 3 month rolling mean (shift back one month to avoid including current month in mean)
weather_monthly_sorted['temp_anom_z_roll3'] = weather_monthly_sorted.groupby('state')['temp_anom_z'].transform(
    lambda x: x.shift(1).rolling(3).mean()
)
weather_monthly_sorted['precip_anom_z_roll3'] = weather_monthly_sorted.groupby('state')['precip_anom_z'].transform(
    lambda x: x.shift(1).rolling(3).mean()
)
weather_monthly_sorted['n_hot_days_roll3'] = weather_monthly_sorted.groupby('state')['n_hot_days'].transform(
    lambda x: x.shift(1).rolling(3).mean()
)
weather_monthly_sorted['drought_index_roll3'] = weather_monthly_sorted.groupby('state')['drought_index'].transform(
    lambda x: x.shift(1).rolling(3).mean()
)
weather_monthly_sorted['heavy_rain_days_roll3'] = weather_monthly_sorted.groupby('state')['heavy_rain_days'].transform(
    lambda x: x.shift(1).rolling(3).mean()
)
# 6 month rolling mean (shift back one month to avoid including current month in mean)
weather_monthly_sorted['temp_anom_z_roll6'] = weather_monthly_sorted.groupby('state')['temp_anom_z'].transform(
    lambda x: x.shift(1).rolling(6).mean()
)
weather_monthly_sorted['precip_anom_z_roll6'] = weather_monthly_sorted.groupby('state')['precip_anom_z'].transform(
    lambda x: x.shift(1).rolling(6).mean()
)
weather_monthly_sorted['n_hot_days_roll6'] = weather_monthly_sorted.groupby('state')['n_hot_days'].transform(
    lambda x: x.shift(1).rolling(6).mean()
)
weather_monthly_sorted['drought_index_roll6'] = weather_monthly_sorted.groupby('state')['drought_index'].transform(
    lambda x: x.shift(1).rolling(6).mean()
)
weather_monthly_sorted['heavy_rain_days_roll6'] = weather_monthly_sorted.groupby('state')['heavy_rain_days'].transform(
    lambda x: x.shift(1).rolling(6).mean()
)

In [14]:
weather_monthly_sorted.head(5)

,state,monthyear,year,month,mean_temp,max_temp,total_precip,n_hot_days,heavy_rain_days,baseline_temp,...,temp_anom_z_roll3,precip_anom_z_roll3,n_hot_days_roll3,drought_index_roll3,heavy_rain_days_roll3,temp_anom_z_roll6,precip_anom_z_roll6,n_hot_days_roll6,drought_index_roll6,heavy_rain_days_roll6
192,Baden-Württemberg,7_2003,2003,7,19.441086,31.734499,56.400002,2,1,19.108015,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
214,Baden-Württemberg,8_2003,2003,8,23.156878,34.684498,18.400000,12,0,18.709295,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
236,Baden-Württemberg,9_2003,2003,9,14.583459,27.684500,25.699999,0,0,14.722257,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,Baden-Württemberg,10_2003,2003,10,6.437322,20.734499,91.799995,0,2,10.330837,...,0.782755,-1.324068,4.666667,2.106823,0.333333,NaN,NaN,NaN,NaN,NaN
22,Baden-Württemberg,11_2003,2003,11,5.571653,15.334500,58.000000,0,1,5.168454,...,-0.003034,-0.684893,4.000000,0.681860,0.666667,NaN,NaN,NaN,NaN,NaN


Remove 2003 data (was only used to calculate lagged variables)

In [15]:
weather_monthly_sorted = weather_monthly_sorted.loc[weather_monthly_sorted['year']>2003]
weather_monthly_sorted.head()

,state,monthyear,year,month,mean_temp,max_temp,total_precip,n_hot_days,heavy_rain_days,baseline_temp,...,temp_anom_z_roll3,precip_anom_z_roll3,n_hot_days_roll3,drought_index_roll3,heavy_rain_days_roll3,temp_anom_z_roll6,precip_anom_z_roll6,n_hot_days_roll6,drought_index_roll6,heavy_rain_days_roll6
66,Baden-Württemberg,1_2004,2004,1,-0.184384,10.834500,133.100006,0,4,0.888582,...,-0.782721,0.099744,0.0,-0.882465,1.333333,0.000017,-0.612162,2.333333,0.612179,0.833333
87,Baden-Württemberg,2_2004,2004,2,1.395922,14.384501,35.500000,0,2,1.658827,...,-0.223042,0.216010,0.0,-0.439053,2.000000,-0.113038,-0.234442,2.000000,0.121404,1.333333
108,Baden-Württemberg,3_2004,2004,3,3.290750,20.434500,39.500000,0,0,4.780743,...,-0.346620,0.042150,0.0,-0.388771,2.333333,-0.497580,-0.013787,0.000000,-0.483792,1.666667
129,Baden-Württemberg,4_2004,2004,4,8.687417,21.584499,40.200001,0,1,9.264582,...,-0.452177,0.061562,0.0,-0.513738,2.000000,-0.617449,0.080653,0.000000,-0.698102,1.666667
150,Baden-Württemberg,5_2004,2004,5,11.212121,22.384499,71.699997,0,0,13.113695,...,-0.405941,-0.709163,0.0,0.303222,1.000000,-0.314492,-0.246576,0.000000,-0.067915,1.500000


In [16]:
weather_monthly_sorted.isna().sum()

state                    0
monthyear                0
year                     0
month                    0
mean_temp                0
max_temp                 0
total_precip             0
n_hot_days               0
heavy_rain_days          0
baseline_temp            0
baseline_precip          0
temp_anomaly             0
precip_prop_anomaly      0
temp_anom_z              0
precip_anom_z            0
drought_index            0
temp_anom_z_lag1         0
precip_anom_z_lag1       0
n_hot_days_lag1          0
drought_index_lag1       0
heavy_rain_days_lag1     0
temp_anom_z_lag2         0
precip_anom_z_lag2       0
n_hot_days_lag2          0
drought_index_lag2       0
heavy_rain_days_lag2     0
temp_anom_z_lag3         0
precip_anom_z_lag3       0
n_hot_days_lag3          0
drought_index_lag3       0
heavy_rain_days_lag3     0
temp_anom_z_roll3        0
precip_anom_z_roll3      0
n_hot_days_roll3         0
drought_index_roll3      0
heavy_rain_days_roll3    0
temp_anom_z_roll6        0
p

Save weather monthly data

In [17]:
weather_monthly_sorted.to_parquet('../Data/Processed/weather_monthly.parquet')

### GBIF data

Calculate:
- log of number of observations per state per month (observation effort)
- number of distinct species observed per state per month (species richness) 

In [18]:
import duckdb
duckdb.query("INSTALL parquet;")
duckdb.query("LOAD parquet;")

In [19]:
gbif = '../Data/Processed/gbif_states.parquet'

duckdb.query(f"""
    SELECT *
    FROM read_parquet('{gbif}')
    LIMIT 5    
""")

┌────────────┬───────────────────────┬───────┬───────┬─────────────────┐
│   gbifID   │        species        │ month │ year  │      state      │
│   int64    │        varchar        │ int64 │ int64 │     varchar     │
├────────────┼───────────────────────┼───────┼───────┼─────────────────┤
│ 4508288286 │ Oxygastra curtisii    │     7 │  2013 │ Rheinland-Pfalz │
│ 4508285279 │ Oxygastra curtisii    │     6 │  2014 │ Rheinland-Pfalz │
│ 1950685555 │ Conistra vaccinii     │     4 │  2004 │ Saarland        │
│ 1950685917 │ Leptidea sinapis      │     4 │  2004 │ Saarland        │
│ 1950685708 │ Scoliopteryx libatrix │     4 │  2004 │ Rheinland-Pfalz │
└────────────┴───────────────────────┴───────┴───────┴─────────────────┘

In [20]:
gbif_richness = duckdb.query(f"""
    SELECT state, year, month,
            COUNT(*) as n_obs,
            LOG(COUNT(*)) as log_n_obs,
            COUNT(DISTINCT(species)) as n_species
    FROM read_parquet('{gbif}') 
    GROUP BY state, year, month
""").to_df()

In [21]:
gbif_richness

,state,year,month,n_obs,log_n_obs,n_species
0,Bayern,2024,10,33274,4.522105,1204
1,Bayern,2024,6,79620,4.901022,3353
2,Niedersachsen,2024,8,61022,4.785486,2225
3,Nordrhein-Westfalen,2020,12,18210,4.260310,437
4,Baden-Württemberg,2022,3,28094,4.448614,858
...,...,...,...,...,...,...
4000,Sachsen-Anhalt,2004,2,7,0.845098,2
4001,Saarland,2008,1,3,0.477121,3
4002,Saarland,2008,12,5,0.698970,5
4003,Bremen,2006,10,14,1.146128,4


Save species richness data

In [22]:
gbif_richness.to_parquet('../Data/Processed/gbif_richness.parquet')

---
### Merge weather and species richness datasets

In [23]:
weather_monthly = pd.read_parquet('../Data/Processed/weather_monthly.parquet')

gbif_richness = pd.read_parquet('../Data/Processed/gbif_richness.parquet')

check if they're both the same length

In [24]:
print(f'weather: {weather_monthly.shape}')
print(f'gbif: {gbif_richness.shape}')

weather: (4032, 41)
gbif: (4005, 6)


27 more rows in weather than gbif - investigate

In [25]:
# Suppose df1 is the larger dataframe, df2 is the reference
missing_rows = weather_monthly.merge(
    gbif_richness,
    on=['state', 'month', 'year'],      # columns to match
    how='left',              # keep all rows from df1
    indicator=True
).query('_merge == "left_only"').drop(columns='_merge')

In [26]:
missing_rows

,state,monthyear,year,month,mean_temp,max_temp,total_precip,n_hot_days,heavy_rain_days,baseline_temp,...,drought_index_roll3,heavy_rain_days_roll3,temp_anom_z_roll6,precip_anom_z_roll6,n_hot_days_roll6,drought_index_roll6,heavy_rain_days_roll6,n_obs,log_n_obs,n_species
1018,Bremen,11_2004,2004,11,5.445708,13.268000,68.400002,0,3,6.549984,...,0.305823,0.666667,-0.520278,0.069515,0.000000,-0.589793,1.333333,NaN,NaN,NaN
1022,Bremen,3_2005,2005,3,4.278955,17.468000,50.799999,0,1,5.261207,...,0.500403,1.000000,-0.341545,-0.451563,0.000000,0.110019,1.166667,NaN,NaN,NaN
1026,Bremen,7_2005,2005,7,17.928350,28.718000,116.700005,0,2,18.439034,...,-0.603656,1.333333,-0.316326,-0.109938,0.166667,-0.206388,1.166667,NaN,NaN,NaN
1028,Bremen,9_2005,2005,9,15.742722,28.318001,37.100002,0,1,15.153910,...,-0.807000,1.333333,-0.595100,0.169021,0.166667,-0.764121,1.500000,NaN,NaN,NaN
1035,Bremen,4_2006,2006,4,7.992375,21.218000,70.400002,0,2,9.323503,...,-0.754782,1.333333,-0.681973,-0.557066,0.000000,-0.124907,1.166667,NaN,NaN,NaN
1039,Bremen,8_2006,2006,8,16.386145,25.168001,137.600006,0,3,18.191109,...,1.603507,0.333333,-0.160462,-0.157793,1.000000,-0.002669,1.166667,NaN,NaN,NaN
1040,Bremen,9_2006,2006,9,17.857792,26.768002,17.100000,0,0,15.153910,...,0.773728,1.000000,-0.243906,0.210074,1.000000,-0.453980,1.500000,NaN,NaN,NaN
1046,Bremen,3_2007,2007,3,7.249653,15.768000,61.799999,0,1,5.261207,...,0.633091,0.666667,1.330548,-0.044962,0.000000,1.375510,1.000000,NaN,NaN,NaN
1050,Bremen,7_2007,2007,7,16.971428,31.068001,123.000000,1,1,18.439034,...,0.441801,3.000000,0.898230,0.506958,0.000000,0.391272,2.000000,NaN,NaN,NaN
1051,Bremen,8_2007,2007,8,17.078550,26.918001,58.000000,0,0,18.191109,...,-1.277100,3.333333,0.481342,0.451586,0.166667,0.029756,1.833333,NaN,NaN,NaN


Impute missing values as the mean number of observations & species richness within the same year and state  

First merge weather and gbif data

In [27]:
full_df = weather_monthly.merge(
    gbif_richness,
    on=['state', 'month', 'year'], 
    how='left')

full_df.shape

(4032, 44)

In [28]:
# Fill missing n_obs, log_n_obs, and n_species with the mean per state/month/year
full_df['n_obs'] = full_df['n_obs'].fillna(
    full_df.groupby(['state', 'year'])['n_obs'].transform('mean').round().astype('Int64')
)
full_df['log_n_obs'] = full_df['log_n_obs'].fillna(
    full_df.groupby(['state', 'year'])['log_n_obs'].transform('mean')
)
full_df['n_species'] = full_df['n_species'].fillna(
    full_df.groupby(['state', 'year'])['n_species'].transform('mean').round().astype('Int64')
)



In [29]:
full_df.loc[(full_df['state']=='Saarland') & (full_df['month']==4) & (full_df['year']==2006)]

,state,monthyear,year,month,mean_temp,max_temp,total_precip,n_hot_days,heavy_rain_days,baseline_temp,...,drought_index_roll3,heavy_rain_days_roll3,temp_anom_z_roll6,precip_anom_z_roll6,n_hot_days_roll6,drought_index_roll6,heavy_rain_days_roll6,n_obs,log_n_obs,n_species
2799,Saarland,4_2006,2006,4,8.679847,19.837,63.100002,0,1,9.976909,...,-0.947929,2.0,-0.67584,-0.409327,0.0,-0.266512,1.666667,282.0,1.541736,51.0


Create binary 'biodiversity anomaly' outcome feature for classification model

1. compute baseline for state and month

In [30]:
# import pandas as pd
# full_df = pd.read_parquet('../Data/Processed/full_df.parquet')

In [31]:
baseline = full_df.groupby(['state','month']).agg(
    mean_richness=('n_species','mean'),
    std_richness=('n_species','std')
).reset_index()

full_df = full_df.merge(baseline, on=['state','month'])

full_df.head()

,state,monthyear,year,month,mean_temp,max_temp,total_precip,n_hot_days,heavy_rain_days,baseline_temp,...,temp_anom_z_roll6,precip_anom_z_roll6,n_hot_days_roll6,drought_index_roll6,heavy_rain_days_roll6,n_obs,log_n_obs,n_species,mean_richness,std_richness
0,Baden-Württemberg,1_2004,2004,1,-0.184384,10.834500,133.100006,0,4,0.888582,...,0.000017,-0.612162,2.333333,0.612179,0.833333,355.0,2.550228,93.0,231.428571,114.674135
1,Baden-Württemberg,2_2004,2004,2,1.395922,14.384501,35.500000,0,2,1.658827,...,-0.113038,-0.234442,2.000000,0.121404,1.333333,274.0,2.437751,108.0,265.761905,187.636858
2,Baden-Württemberg,3_2004,2004,3,3.290750,20.434500,39.500000,0,0,4.780743,...,-0.497580,-0.013787,0.000000,-0.483792,1.666667,815.0,2.911158,176.0,438.238095,290.009121
3,Baden-Württemberg,4_2004,2004,4,8.687417,21.584499,40.200001,0,1,9.264582,...,-0.617449,0.080653,0.000000,-0.698102,1.666667,658.0,2.818226,197.0,743.523810,497.637581
4,Baden-Württemberg,5_2004,2004,5,11.212121,22.384499,71.699997,0,0,13.113695,...,-0.314492,-0.246576,0.000000,-0.067915,1.500000,914.0,2.960946,223.0,1135.380952,754.805039


2. Control for observation effort and year trend (with regression residuals)

In [32]:
# import statsmodels.api as sm

# X = sm.add_constant(full_df['log_n_obs']) # Creates dataframe of 1s (intercept) and log observations
# model = sm.OLS(full_df['n_species'], X).fit()

# full_df['expected_richness'] = model.predict(X)
# full_df['residual'] = full_df['n_species'] - full_df['expected_richness']

In [ ]:
import statsmodels.api as sm

# Create design matrix
X = pd.get_dummies(full_df['state'], drop_first=True)

X['log_n_obs'] = full_df['log_n_obs']
X['year'] = full_df['year']

X = X.astype(float)

# Add intercept
X = sm.add_constant(X)

# Fit model
model = sm.OLS(full_df['n_species'], X).fit()

3. Standardise residuals per state and month

In [56]:
full_df['residual_z'] = full_df.groupby(['state','month'])['residual'].transform(
    lambda x: (x - x.mean()) / x.std()
)

4. Define binary anomaly  

interpretation:  
1 = biodiversity is unusually LOW for this state/month, given effort  
0 = normal or high biodiversity  

(or fine tune the threshold:  
-0.5 → more sensitive  
-1.0 → standard  
-1.5 → only extreme events
)

In [57]:
full_df['biodiversity_anomaly_standard'] = (full_df['residual_z'] < -1).astype(int)
full_df['biodiversity_anomaly_sensitive'] = (full_df['residual_z'] < -0.5).astype(int)
full_df['biodiversity_anomaly_conservative'] = (full_df['residual_z'] < -1.5).astype(int)

In [58]:
full_df.head()

,state,monthyear,year,month,mean_temp,max_temp,total_precip,n_hot_days,heavy_rain_days,baseline_temp,...,log_n_obs,n_species,mean_richness,std_richness,expected_richness,residual,residual_z,biodiversity_anomaly_standard,biodiversity_anomaly_sensitive,biodiversity_anomaly_conservative
0,Baden-Württemberg,1_2004,2004,1,-0.184384,10.834500,133.100006,0,4,0.888582,...,2.550228,93.0,231.428571,114.674135,152.419101,-59.419101,1.470503,0,0,0
1,Baden-Württemberg,2_2004,2004,2,1.395922,14.384501,35.500000,0,2,1.658827,...,2.437751,108.0,265.761905,187.636858,112.771138,-4.771138,1.424803,0,0,0
2,Baden-Württemberg,3_2004,2004,3,3.290750,20.434500,39.500000,0,0,4.780743,...,2.911158,176.0,438.238095,290.009121,279.645196,-103.645196,0.240203,0,0,0
3,Baden-Württemberg,4_2004,2004,4,8.687417,21.584499,40.200001,0,1,9.264582,...,2.818226,197.0,743.523810,497.637581,246.887144,-49.887144,-0.493433,0,0,0
4,Baden-Württemberg,5_2004,2004,5,11.212121,22.384499,71.699997,0,0,13.113695,...,2.960946,223.0,1135.380952,754.805039,297.195470,-74.195470,-0.718204,0,1,0


In [60]:
full_df.value_counts('biodiversity_anomaly_standard')

biodiversity_anomaly_standard
0    3733
1     299
Name: count, dtype: int64

In [61]:
full_df.value_counts('biodiversity_anomaly_sensitive')

biodiversity_anomaly_sensitive
0    2423
1    1609
Name: count, dtype: int64

In [62]:
full_df.value_counts('biodiversity_anomaly_conservative')

biodiversity_anomaly_conservative
0    4007
1      25
Name: count, dtype: int64

Double check that lagged variable calculations worked correctly

In [63]:
full_df.loc[full_df['state'].isin(['Bayern', 'Berlin'])][['state', 'monthyear', 'temp_anom_z', 'temp_anom_z_lag1']].groupby('state').head(3)

,state,monthyear,temp_anom_z,temp_anom_z_lag1
252,Bayern,1_2004,-0.484201,-0.255672
253,Bayern,2_2004,0.256806,-0.484201
254,Bayern,3_2004,-0.746567,0.256806
504,Berlin,1_2004,-0.731760,-0.016205
505,Berlin,2_2004,0.282089,-0.731760
506,Berlin,3_2004,0.109810,0.282089


Looks good - check if Jan lagged values are from December in previous year

In [64]:
(full_df.loc[(full_df['state'].isin(['Bayern', 'Berlin'])) 
            & (full_df['monthyear'].isin(['12_2004', '1_2005']))]
            [['state', 'monthyear', 'temp_anom_z', 'temp_anom_z_lag1']]
            .groupby('state')
            .head(3))

,state,monthyear,temp_anom_z,temp_anom_z_lag1
263,Bayern,12_2004,-0.890507,-0.991200
264,Bayern,1_2005,0.008640,-0.890507
515,Berlin,12_2004,-0.136432,-0.731508
516,Berlin,1_2005,0.646846,-0.136432


Looks good!

Circular encoding of months  
(so that December (12) and January (1) are not interpreted as 11 months apart from each other)  

What This Achieves  
1. Cyclic continuity:  
- December → January is smooth, not “jumping” from 12 → 1  
2. Numeric representation:  
- Each month gets two numeric features  
- Model can now learn patterns like “winter months” or “summer months” continuously  
3. Avoids artificial ordering:  
- Tree-based models or linear models now handle seasonality correctly  

In [65]:
from numpy import pi, sin, cos

full_df['month_sin'] = sin(2*pi * full_df['month'] / 12)
full_df['month_cos'] = cos(2*pi * full_df['month'] / 12)

Now re-base the year to 0

In [66]:
full_df['year_offset'] = full_df['year'] - min(full_df['year'])

In [67]:
full_df

,state,monthyear,year,month,mean_temp,max_temp,total_precip,n_hot_days,heavy_rain_days,baseline_temp,...,std_richness,expected_richness,residual,residual_z,biodiversity_anomaly_standard,biodiversity_anomaly_sensitive,biodiversity_anomaly_conservative,month_sin,month_cos,year_offset
0,Baden-Württemberg,1_2004,2004,1,-0.184384,10.834500,133.100006,0,4,0.888582,...,114.674135,152.419101,-59.419101,1.470503,0,0,0,5.000000e-01,8.660254e-01,0
1,Baden-Württemberg,2_2004,2004,2,1.395922,14.384501,35.500000,0,2,1.658827,...,187.636858,112.771138,-4.771138,1.424803,0,0,0,8.660254e-01,5.000000e-01,0
2,Baden-Württemberg,3_2004,2004,3,3.290750,20.434500,39.500000,0,0,4.780743,...,290.009121,279.645196,-103.645196,0.240203,0,0,0,1.000000e+00,6.123234e-17,0
3,Baden-Württemberg,4_2004,2004,4,8.687417,21.584499,40.200001,0,1,9.264582,...,497.637581,246.887144,-49.887144,-0.493433,0,0,0,8.660254e-01,-5.000000e-01,0
4,Baden-Württemberg,5_2004,2004,5,11.212121,22.384499,71.699997,0,0,13.113695,...,754.805039,297.195470,-74.195470,-0.718204,0,1,0,5.000000e-01,-8.660254e-01,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4027,Thüringen,8_2024,2024,8,19.984138,30.950001,72.899994,2,2,18.361685,...,243.336305,655.991629,220.008371,1.775052,0,0,0,-8.660254e-01,-5.000000e-01,20
4028,Thüringen,9_2024,2024,9,15.468820,30.500000,88.400002,1,2,14.538055,...,160.238676,588.713105,-2.713105,0.772059,0,0,0,-1.000000e+00,-1.836970e-16,20
4029,Thüringen,10_2024,2024,10,11.061694,18.700001,62.299999,0,1,10.015421,...,117.854147,551.796588,-140.796588,-0.364042,0,0,0,-8.660254e-01,5.000000e-01,20
4030,Thüringen,11_2024,2024,11,4.483958,13.750000,53.499996,0,1,5.154050,...,62.805179,500.806827,-272.806827,-0.782543,0,1,0,-5.000000e-01,8.660254e-01,20


Save full_df - ready for model pipeline

In [68]:
full_df.to_parquet('../Data/Processed/full_df.parquet')

In [69]:
full_df.sample(10)

,state,monthyear,year,month,mean_temp,max_temp,total_precip,n_hot_days,heavy_rain_days,baseline_temp,...,std_richness,expected_richness,residual,residual_z,biodiversity_anomaly_standard,biodiversity_anomaly_sensitive,biodiversity_anomaly_conservative,month_sin,month_cos,year_offset
3831,Thüringen,4_2008,2008,4,7.191181,19.972500,89.199997,0,2,8.713919,...,150.289483,93.125299,17.874701,0.609310,0,0,0,8.660254e-01,-5.000000e-01,4
3448,Sachsen-Anhalt,5_2018,2018,5,17.366129,30.700001,21.400000,2,0,14.080561,...,348.083597,646.287257,-75.287257,0.117900,0,0,0,5.000000e-01,-8.660254e-01,14
1697,Hessen,6_2019,2019,6,19.982679,34.389000,66.699997,3,2,17.087580,...,638.508508,731.602022,649.397978,0.280380,0,0,0,1.224647e-16,-1.000000e+00,15
1485,Hamburg,10_2022,2022,10,12.934812,21.200001,38.799999,0,1,10.678127,...,95.845436,390.837322,-148.837322,-0.922524,0,1,0,-8.660254e-01,5.000000e-01,18
3057,Sachsen,10_2006,2006,10,12.200134,19.975000,78.100006,0,3,10.117830,...,244.810772,182.825239,-52.825239,-1.147314,1,1,0,-8.660254e-01,5.000000e-01,2
2085,Niedersachsen,10_2009,2009,10,9.039613,19.763000,80.700005,0,2,11.116194,...,349.380493,542.267514,-145.267514,-0.472997,0,0,0,-8.660254e-01,5.000000e-01,5
2858,Saarland,3_2011,2011,3,6.678264,16.987000,17.500000,0,0,5.812680,...,70.520412,-220.827089,245.827089,0.685099,0,0,0,1.000000e+00,6.123234e-17,7
1695,Hessen,4_2019,2019,4,10.190528,22.639000,49.200001,0,2,9.305956,...,400.763189,774.806385,103.193615,0.441365,0,0,0,8.660254e-01,-5.000000e-01,15
1919,Mecklenburg-Vorpommern,12_2016,2016,12,3.573449,9.859000,45.200001,0,1,2.689586,...,51.731311,296.187209,-157.187209,-0.045095,0,0,0,-2.449294e-16,1.000000e+00,12
3729,Schleswig-Holstein,10_2020,2020,10,10.879906,17.750000,81.599998,0,0,10.562412,...,182.179124,599.336538,-176.336538,-0.485232,0,0,0,-8.660254e-01,5.000000e-01,16
